In [ ]:
import re
import os
import requests
import html2text
import time
from bs4 import BeautifulSoup
from collections import deque

pattern = r"https?:\/\/(?:www\d*\.)?eecs\.berkeley\.edu(?:\/[^\s]*)?"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

os.makedirs("html", exist_ok=True)
os.makedirs("cleaned_text", exist_ok=True)

h2t = html2text.HTML2Text()
h2t.ignore_links = True
h2t.ignore_images = True
h2t.ignore_tables = False
h2t.body_width = 0


def normalize_url(url):
    """Strip fragment and trailing slash for deduplication."""
    url = url.split("#")[0].rstrip("/")
    return url


def url_to_safe_name(url):
    url = normalize_url(url)
    url = url.replace("http://", "").replace("https://", "")
    return re.sub(r"[^\w-]", "_", url)


def get_matching_urls(html, base_url=None):
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return []

    urls = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if href.startswith("/") and base_url:
            href = base_url.rstrip("/") + href

        href = normalize_url(href)

        if re.match(pattern, href):
            urls.append(href)

    return urls

In [ ]:
seed = "https://eecs.berkeley.edu"
max_pages = 100000
retry_delay = 5   # seconds to wait on first 429; doubles each consecutive hit

# --- Restart: rebuild visited + queue from already-saved HTML files ---
visited = set()
queue = deque()

print("Scanning existing html/ for already-crawled pages...")
for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue
    safe_name = filename[:-5]
    visited.add(safe_name)

# Re-read saved HTML to reconstruct the link frontier.
for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue
    with open(f"html/{filename}") as f:
        html = f.read()
    safe_name = filename[:-5]
    for u in get_matching_urls(html):
        if url_to_safe_name(u) not in visited:
            queue.append(u)

# If nothing was found (fresh start), begin from seed.
if not visited and not queue:
    queue.append(seed)

print(f"Resuming: {len(visited)} pages already crawled, {len(queue)} URLs queued.\n")
# ---------------------------------------------------------------------

backoff = retry_delay  # tracks current wait time for 429s

while queue and len(visited) < max_pages:
    url = queue.popleft()
    safe_name = url_to_safe_name(url)

    if safe_name in visited:
        continue

    print(f"Visiting ({len(visited)}/{max_pages}): {url}")

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
    except Exception as e:
        print(f"  Request error: {e} — skipping.")
        visited.add(safe_name)
        continue

    # 429 Too Many Requests — back off and retry
    if response.status_code == 429:
        print(f"  429 Too Many Requests — waiting {backoff}s before retrying...")
        time.sleep(backoff)
        backoff = min(backoff * 2, 120)  # exponential backoff, cap at 2 min
        queue.appendleft(url)            # put it back at the front
        continue

    # Reset backoff after a successful request
    backoff = retry_delay

    # 404 or other non-2xx — skip permanently
    if response.status_code == 404:
        print(f"  404 Not Found — skipping.")
        visited.add(safe_name)
        continue

    if not response.ok:
        print(f"  HTTP {response.status_code} — skipping.")
        visited.add(safe_name)
        continue

    # Cloudflare challenge page
    if "Just a moment..." in response.text:
        print(f"  Cloudflare challenge — waiting {backoff}s before retrying...")
        time.sleep(backoff)
        backoff = min(backoff * 2, 120)
        queue.appendleft(url)
        continue

    if "Forbidden" in response.text:
        visited.add(safe_name)
        continue

    visited.add(safe_name)
    with open(f"html/{safe_name}.html", "w") as f:
        f.write(response.text)

    new_urls = get_matching_urls(response.text, base_url=url)
    for u in new_urls:
        if url_to_safe_name(u) not in visited:
            queue.append(u)

print(f"\nDone. Crawled {len(visited)} pages.")

In [ ]:
def clean_html_for_rag(html):
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return ""

    for tag in soup([
        "script", "style", "noscript",
        "nav", "footer", "header",
        "form", "input", "button", "select", "textarea",
        "svg", "img"
    ]):
        tag.decompose()

    noise_classes = [
        "sidebar", "ad", "ads", "promo",
        "cookie", "banner", "popup",
        "modal", "social", "share",
        "related", "footer"
    ]

    for tag in soup.find_all(True):
        classes = " ".join(tag.attrs.get("class", [])) if tag.attrs else ""
        if any(n in classes.lower() for n in noise_classes):
            tag.decompose()

    return h2t.handle(str(soup))


for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue

    with open(f"html/{filename}") as f:
        html = f.read()

    text = clean_html_for_rag(html)

    out_name = filename[:-5]  # strip .html
    with open(f"cleaned_text/{out_name}.txt", "w") as f:
        f.write(text)

print(f"Done. Converted {len(os.listdir('html'))} files.")